# Provenance tutorial
This tutorial will show you how to:
- understand `Provenance` objects and their role in DNAStream. 
- using Python special methods on a `Provenance` object
- query/export the provenance log or a subset to a `pandas.Dataframe`. 


## `Provenance objects and their role in DNAStream
Since research is an iterative and exploratory process. DNAStream was designed to facilitate one or more uses to complete many upstream and downstream analyses on mult-modal DNA sequencing data. Subsequently, it becomes important to track any modifications to the DNAStream HDF5 file. Provenance datasets are used to track these modifications.  Currently, there is only one built-in provenance dataset name 'log'.  Similarily to `Registry`, DNAStream provides a specialized `Provenance` interface to a provenance dataset. 

Via `Provenance` objects, one can:   
- extract the provenance log via `to_dataframe`
- iterate through the provenance log dataset
- determine the number of logged modifications

### Provenance log fields

The provenance log contains the the following fields:
- id : unique identifier of the event (UUIDv4)
- timestamp : time and data of when the modification occurred (ISO8601 Z)
- user : the user that performed the mofidication
- scope : which interface (DNAStream, IO, Registry, etc) performed the modification
- event : type of modification; one of create, append, modify, designate
- dataset : the dataset in the HDF5 file where the modification occurred
- source : the string name of the function that was used to modify the file
- info : a JSON string that includes amplifying information about the modification

In [1]:

from pathlib import Path
import tempfile
from dnastream import DNAStream

tmpdir = Path(tempfile.mkdtemp(prefix="dnastream_tutorial_"))
myfile = tmpdir / "temp.h5"

# Create the DNAStream file (fails if it already exists)
ds = DNAStream.create(path=str(myfile), patient_id="patientX", verbose=True)
ds.close()

print("Created:", myfile)

2026-02-05 13:21:01 - INFO - Created DNAStream file /var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_2vc8ddod/temp.h5
2026-02-05 13:21:01 - INFO - Connection to DNAStream file '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_2vc8ddod/temp.h5' closed.


Created: /var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_2vc8ddod/temp.h5


## Provenance log
The built-in provenance modfication log can be acesssed in two ways, by attribute or by a `provenance(name)` method.

In [2]:
with DNAStream.open(myfile) as ds:

    print(ds.log)
    log = ds.provenance("log")
    assert ds.log == log

/provenance/log (Shape: (4,))


## Using Python special methods on a `Provenance` object
The `Provenance` interface provides the following built Python special methods:
- `__iter__` : iterate through each logged modification in the provenance dataset
- `__len__` : get the total number of entries in the provenance dataset.

In [3]:
with DNAStream.open(myfile) as ds:
    nevents = len(ds.log)
    print(f"The modification log contains {nevents} events")

    for mod in ds.log:
        if mod["event"] == "create":
            print(mod)

The modification log contains 4 events
{'id': 'f7242c47-a32d-47e6-afae-20459af18a69', 'timestamp': '2026-02-05T19:21:01.819810Z', 'user': 'llweber', 'scope': 'DNAStream', 'event': 'create', 'dataset': '/registry/sample', 'source': 'dnastream.registry.Registry.create', 'info': '{"schema_hash": "1090178384a29f5b1a363d720fcd1c1ca051f02e8682c36e78144250a8685a0d", "schema_version": "0.1.0"}'}
{'id': 'd236558f-0de0-42eb-a307-de702d92ea67', 'timestamp': '2026-02-05T19:21:01.820674Z', 'user': 'llweber', 'scope': 'DNAStream', 'event': 'create', 'dataset': '/registry/variant', 'source': 'dnastream.registry.Registry.create', 'info': '{"schema_hash": "73d79ee728f843163e4a7436fec1bff8c77d4ca1c4083aea44ab4a5f2ff1b488", "schema_version": "0.1.0"}'}
{'id': '81d61214-4cce-4da8-887c-23c610134073', 'timestamp': '2026-02-05T19:21:01.821325Z', 'user': 'llweber', 'scope': 'DNAStream', 'event': 'create', 'dataset': '/registry/snp', 'source': 'dnastream.registry.Registry.create', 'info': '{"schema_hash": "904

## Query/export the provenance log or a subset to a `pandas.Dataframe`.
Users can export views of a proveance dataset as a `pandas.Dataframe` via  the `to_dataframe(...)` function. **Note, these only return copies of the dataset as a dataframe and any modifications of this dataframe will not be reflected in the DNAStream HDF5 file. ** 

In [4]:
with DNAStream.open(myfile) as ds:

    log_df = ds.log.to_dataframe()

    print(log_df)

                                     id                    timestamp     user  \
0  f7242c47-a32d-47e6-afae-20459af18a69  2026-02-05T19:21:01.819810Z  llweber   
1  d236558f-0de0-42eb-a307-de702d92ea67  2026-02-05T19:21:01.820674Z  llweber   
2  81d61214-4cce-4da8-887c-23c610134073  2026-02-05T19:21:01.821325Z  llweber   
3  9d0432f4-aaad-4b21-9d58-865687af3954  2026-02-05T19:21:01.824047Z  llweber   

       scope   event            dataset                                source  \
0  DNAStream  create   /registry/sample    dnastream.registry.Registry.create   
1  DNAStream  create  /registry/variant    dnastream.registry.Registry.create   
2  DNAStream  create      /registry/snp    dnastream.registry.Registry.create   
3  DNAStream  create                  .  dnastream.dnastream.DNAStream.create   

                                                info  
0  {"schema_hash": "1090178384a29f5b1a363d720fcd1...  
1  {"schema_hash": "73d79ee728f843163e4a7436fec1b...  
2  {"schema_hash": "904